---
title: "Obtaining Edges using the Canny operator"
description: "In this tutorial we show how easily one can apply the typical canny edge detection using Kornia"
author:
    - "Pau Riba"
date: 06-08-2021
categories:
    - Basic
    - Edge Detection
    - kornia.filters
image: "../tutorials/assets/canny.png"
---

<a href="https://colab.sandbox.google.com/github/kornia/tutorials/blob/master/nbs/canny.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in google colab"></a>

## Preparation

We first install Kornia.

In [ ]:
%%capture
%matplotlib inline
!pip install kornia
!pip install kornia-rs

In [ ]:
import kornia
import numpy as np
from PIL import Image

kornia.__version__

Now we download the example image.

In [ ]:
import io

import requests


def download_image(url: str, filename: str = "") -> str:
    filename = url.split("/")[-1] if len(filename) == 0 else filename
    # Download
    bytesio = io.BytesIO(requests.get(url).content)
    # Save file
    with open(filename, "wb") as outfile:
        outfile.write(bytesio.getbuffer())

    return filename


url = "https://github.com/kornia/data/raw/main/paranoia_agent.jpg"
download_image(url)

## Example

We first import the required libraries and load the data.

In [ ]:
import kornia
import matplotlib.pyplot as plt
import torch

# read the image with Kornia
img_tensor = (kornia.image_to_tensor(np.array(Image.open("paranoia_agent.jpg").convert("RGB"))).float() / 255.0)[
    None, ...
]  # BxCxHxW
img_array = kornia.tensor_to_image(img_tensor)

plt.axis("off")
plt.imshow(img_array)
plt.show()

To apply a filter, we create the Canny operator object and apply it to the data. It will provide the magnitudes as well as the edges after the hysteresis process. Note that the edges are a binary image which is not differentiable!

In [ ]:
# create the operator
canny = kornia.filters.Canny()

# blur the image
x_magnitude, x_canny = canny(img_tensor)

That's it! We can compare the source image and the results from the magnitude as well as the edges:

In [ ]:
# convert back to numpy
img_magnitude = kornia.tensor_to_image(x_magnitude.byte())
img_canny = kornia.tensor_to_image(x_canny.byte())

# Create the plot
fig, axs = plt.subplots(1, 3, figsize=(16, 16))
axs = axs.ravel()

axs[0].axis("off")
axs[0].set_title("image source")
axs[0].imshow(img_array)

axs[1].axis("off")
axs[1].set_title("canny magnitude")
axs[1].imshow(img_magnitude, cmap="Greys")

axs[2].axis("off")
axs[2].set_title("canny edges")
axs[2].imshow(img_canny, cmap="Greys")

plt.show()

Note that our final result still recovers some edges whose magnitude is quite low. Let us increase the thresholds and compare the final edges.

In [ ]:
# create the operator
canny = kornia.filters.Canny(low_threshold=0.4, high_threshold=0.5)

# blur the image
_, x_canny_threshold = canny(img_tensor)

In [ ]:
import torch.nn.functional as F

# convert back to numpy
img_canny_threshold = kornia.tensor_to_image(x_canny_threshold.byte())

# Create the plot
fig, axs = plt.subplots(1, 3, figsize=(16, 16))
axs = axs.ravel()

axs[0].axis("off")
axs[0].set_title("image source")
axs[0].imshow(img_array)

axs[1].axis("off")
axs[1].set_title("canny default")
axs[1].imshow(img_canny, cmap="Greys")

axs[2].axis("off")
axs[2].set_title("canny defined thresholds")
axs[2].imshow(img_canny_threshold, cmap="Greys")

plt.show()